# Saída Estruturada

Um modelo de linguagem produz texto. A aplicação que o chama precisa de valores tipados: um rótulo de um conjunto fechado, um booleano, um inteiro dentro de uma faixa. Saída estruturada é o conjunto de técnicas que atravessa essa fronteira.

O notebook percorre essas técnicas em ordem crescente de garantia, que também é a ordem em que convém tentá-las. Pedir o formato no prompt é barato e não promete nada. Declarar o contrato como uma classe dá diagnóstico, e diagnóstico não conserta. Devolver o erro ao modelo conserta parte das falhas, ao custo de uma chamada a mais. Restringir a geração token a token torna a saída fora do esquema irrepresentável.

Nenhuma delas alcança o conteúdo: validade e verdade são propriedades separadas.

Uma arrumação alternativa das mesmas técnicas é pelo momento em que cada uma atua, antes da geração, depois dela ou durante.

In [ ]:
# No Google Colab, descomente e rode uma vez (Ambiente de execução > GPU).
# !pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

import json

import pandas as pd
import torch
from pydantic import BaseModel, Field, ValidationError
from typing import Literal

from agentkit import LLM

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
llm = LLM(MODEL_NAME, device=device, temperature=0.0, max_tokens=160)
print(llm.model)

## Extração sem contrato

A tarefa é extrair três campos de uma resenha de produto. Os três têm tipos diferentes entre si, o que faz aparecerem falhas diferentes: o sentimento vem de um conjunto fechado, a recomendação é um booleano e a nota é um inteiro em uma faixa.

In [ ]:
review = (
    "Comprei esta mochila para uma viagem longa e usei todos os dias. "
    "Nada rasgou e ela continua com cara de nova. Dou nota 9 de 10 e compraria de novo."
)
target = {"sentiment": "positivo", "recommends": True, "rating": 9}
target

O primeiro pedido descreve os campos em uma frase e pede JSON.

In [ ]:
raw = llm.invoke([
    {"role": "user", "content": (
        "Extraia o sentimento, se o cliente recomenda o produto "
        f"e a nota desta resenha em JSON:\n\n{review}"
    )},
])
print(raw)

In [ ]:
try:
    json.loads(raw)
except json.JSONDecodeError as error:
    print("json.JSONDecodeError:", error)

O conteúdo está certo e o programa não consegue usá-lo. São três problemas distintos. A moldura em volta do objeto, que aqui é a cerca de bloco de código. Os nomes dos campos, que o pedido não declarou e o modelo escolheu. E o tipo dos valores, que em outras execuções vem como texto no lugar de booleano.

Recortar o texto entre a primeira e a última chave resolveria o primeiro problema e nenhum dos outros dois, porque eles não são de formato: são de contrato. É por aí que o notebook segue.

## Contrato como classe

Contrato aqui é a lista do que a aplicação exige da resposta: quais campos existem, de que tipo é cada um e quais valores são aceitos. Ele pode ser escrito à mão, como uma sequência de `if` que confere o dicionário campo a campo. Nesse formato ele fica espalhado pelo código, e a primeira alteração já deixa alguma checagem para trás.

Escrito como uma classe, o mesmo contrato fica num lugar só e faz três trabalhos ao mesmo tempo: confere o dado que chegou, documenta o que a aplicação espera e produz a mensagem de erro quando algo não bate.

Em Python isso se faz com Pydantic. A classe herda de `BaseModel`, cada campo é uma anotação de tipo, `Field` acrescenta faixa de valores e `Literal` fecha o conjunto permitido. A checagem acontece com o programa rodando, que é o único momento possível: o dado vem de fora.

In [ ]:
class Review(BaseModel):
    sentiment: Literal["positivo", "neutro", "negativo"]
    recommends: bool
    rating: int = Field(ge=0, le=10)

In [ ]:
parsed = Review.model_validate(target)
print(parsed.rating + 1)
print(parsed.model_dump())

O retorno é um objeto: os campos ficam acessíveis por atributo, com tipo garantido, e o caminho de volta ao dicionário continua disponível. Os nomes dos campos ficam em inglês, como todo identificador do código, e os valores do conjunto fechado ficam em português, porque é isso que o modelo escreve a partir de um prompt em português.

A resposta do modelo, validada direto do texto que ele produziu, não passa.

In [ ]:
try:
    Review.model_validate_json(raw)
except ValidationError as error:
    print(error)

A mensagem nomeia o que falhou, em texto que pode ser inserido em um prompt sem tratamento, e é essa propriedade que a correção por reprompt usa adiante. Campos fora do esquema, por outro lado, passam sem reclamação: `model_config = ConfigDict(extra="forbid")` faz a validação recusá-los, e a escolha entre as duas depende de quem produz o dado.

## Contrato no prompt

O pedido original não declarou nem o formato nem os nomes dos campos, e o modelo escolheu os dois. A instrução abaixo declara o contrato dentro do prompt.

In [ ]:
INSTRUCTION = """Extraia estes campos da resenha e responda apenas com JSON.
sentiment: um de "positivo", "neutro", "negativo"
recommends: true ou false
rating: inteiro de 0 a 10"""
print(INSTRUCTION)

Essa instrução repete à mão o que a classe já diz, e duas escritas do mesmo contrato divergem na primeira alteração. O Pydantic evita isso gerando o esquema a partir da própria classe.

In [ ]:
SCHEMA_INSTRUCTION = (
    "Extraia os campos da resenha e responda apenas com JSON, "
    "seguindo este JSON Schema:\n" + json.dumps(Review.model_json_schema())
)
print(SCHEMA_INSTRUCTION)

A comparação entre as três versões precisa de mais de uma execução, porque a falha é intermitente. As amostras usam temperatura acima de zero, que é a condição em que o problema aparece na prática, e a validação é feita direto sobre a resposta, de modo que a moldura conta como falha.

In [ ]:
def validity_rate(prompt: str, samples: int = 4) -> float:
    """Mede a fração de amostras que passam na validação do esquema."""
    valid = 0
    for seed in range(samples):
        torch.manual_seed(seed)
        answer = llm.invoke([{"role": "user", "content": prompt}], temperature=0.7)
        try:
            Review.model_validate_json(answer)
            valid += 1
        except ValidationError:
            pass
    return valid / samples

In [ ]:
loose = f"Extraia o sentimento, a recomendação e a nota desta resenha em JSON:\n\n{review}"
strict = f"{INSTRUCTION}\n\nResenha: {review}"
generated = f"{SCHEMA_INSTRUCTION}\n\nResenha: {review}"
pd.DataFrame(
    {
        "prompt": ["descrição livre", "contrato escrito à mão", "esquema gerado pela classe"],
        "tokens": [len(llm.tokenizer.encode(prompt)) for prompt in [loose, strict, generated]],
        "validity": [validity_rate(loose), validity_rate(strict), validity_rate(generated)],
    }
)

A descrição livre não produz nenhuma amostra válida. Declarar o contrato custa alguns tokens a mais e é a intervenção de menor custo entre as três, mas nesta execução ainda deixa uma amostra fora do contrato; só a versão gerada pela classe passa em todas, e é também a mais cara em tokens e a única que acompanha a classe automaticamente. As duas versões declaradas continuam sendo um pedido: nada na geração impede a saída fora do contrato.

## Correção por reprompt

Declarar o contrato não resolve todos os casos. A resenha abaixo foi escrita para violá-lo: mistura elogio e reclamação, nomeia o próprio sentimento como misto, cita uma nota fora da escala e deixa a recomendação em dúvida.

In [ ]:
hard_review = (
    "Sentimento misto: o tecido é excelente e o zíper é horrível. "
    "Dou nota 12 de 10 para a aparência e 2 de 10 para a durabilidade, e não sei se recomendo."
)
hard_prompt = f"{INSTRUCTION}\n\nResenha: {hard_review}"

In [ ]:
for seed in range(4):
    torch.manual_seed(seed)
    answer = llm.invoke([{"role": "user", "content": hard_prompt}], temperature=0.7)
    try:
        print("válido:  ", Review.model_validate_json(answer))
    except ValidationError as error:
        print("inválido:", error.errors()[0]["msg"], "|", answer.replace("\n", " ")[:60])

As amostras violam o contrato no campo de sentimento, porque o modelo copia do texto uma palavra fora do conjunto permitido, e uma delas nem chega a esse ponto, porque veio embrulhada em cerca de bloco de código. Devolver a mensagem de validação ao modelo em um novo turno é o mecanismo de correção desta parte, e a moldura é a única falha que não vale a pena devolver: recortar entre a primeira e a última chave custa uma linha, e pedir de novo custa uma chamada.

In [ ]:
def generate_valid(prompt: str, schema: type, attempts: int = 3):
    """Repete a chamada devolvendo o erro de validação ao modelo até obter um objeto válido."""
    messages = [{"role": "user", "content": prompt}]
    for attempt in range(1, attempts + 1):
        answer = llm.invoke(messages, temperature=0.7)
        # A cerca de bloco de código é a moldura mais comum, e recortar entre a
        # primeira e a última chave custa uma linha. Gastar uma chamada ao modelo
        # para pedir a remoção dela custa muito mais.
        payload = answer[answer.find("{") : answer.rfind("}") + 1]
        try:
            return schema.model_validate_json(payload), attempt
        except ValidationError as error:
            # A conversa é remontada com a última resposta e o último erro, e as
            # tentativas anteriores são descartadas. Manter o histórico inteiro de
            # erros também é política válida, e cresce em tokens a cada rodada.
            messages = [
                {"role": "user", "content": prompt},
                {"role": "assistant", "content": answer},
                {"role": "user", "content": f"Essa resposta é inválida:\n{error}\n\nResponda de novo, apenas com o JSON corrigido."},
            ]
    return None, attempts

In [ ]:
rows = []
for seed in range(3):
    torch.manual_seed(seed)
    result, attempts = generate_valid(hard_prompt, Review)
    rows.append({"seed": seed, "attempts": attempts, "result": result})
pd.DataFrame(rows)

A coluna `attempts` registra quantas chamadas cada execução precisou, e o retorno é um objeto validado ou `None`. O laço aumenta a taxa de validade sem garanti-la: o modelo pode repetir o mesmo erro nas três tentativas, e o limite existe por isso. O mecanismo também é o único aplicável a regras que o esquema não expressa, como exigir que a nota seja a que o texto cita, e por isso continua em uso depois da geração guiada.

### Exercício 1

Escreva o contrato dos relatos de triagem como uma classe `Patient`, com nome, idade, sintoma principal, há quantos dias os sintomas começaram e a gravidade, que é `leve`, `moderada` ou `grave`. Decida no contrato e na instrução o que acontece quando o relato não diz o número de dias. Escreva `triage`, que declara o contrato no prompt e chama `generate_valid`.

Rode nos cinco relatos e responda quantas tentativas cada um precisou e o que veio no campo de duração dos relatos sem número de dias. Depois aperte o contrato, trocando o sintoma por um conjunto fechado dos cinco sintomas da lista, e responda o que muda no número de tentativas e nos outros campos.

In [ ]:
PATIENTS = [
    "Ana Beatriz, 27, com dor de garganta e febre baixa há três dias.",
    "Paciente Carlos Menezes, 61 anos, falta de ar intensa desde ontem à noite.",
    "Juliana Rocha, 45, dor de cabeça leve que vai e volta há mais ou menos duas semanas.",
    "Menino de 8 anos, Pedro Lima, tosse seca persistente há cinco dias, sem febre.",
    "Marta Souza, 52, dor lombar forte que começou logo depois do carnaval.",
]


def triage(text: str):
    """Extrai o relato de triagem e devolve o objeto validado."""
    return None

## Geração guiada

As técnicas anteriores pedem o formato no prompt e verificam depois da geração. A alternativa é restringir, a cada posição, o conjunto de tokens que podem ser escolhidos, de modo que a saída fora do formato deixe de ser representável.

### Classificação por comparação de logits

O caso mais simples é a classificação, em que a resposta inteira é um rótulo de um conjunto conhecido. Em vez de gerar texto livre e conferir depois, o forward pass é executado uma vez e os logits dos rótulos candidatos são comparados entre si.

In [ ]:
LABELS = ["positivo", "neutro", "negativo"]
# O template do Qwen fecha a abertura do turno do assistente sem espaço pendente,
# então o primeiro token gerado coincide com o primeiro token do rótulo codificado
# sozinho. Em tokenizadores que embutem o espaço no começo da palavra a coincidência
# quebra em silêncio, e a comparação passa a ser entre ids que não correspondem.
label_ids = [llm.tokenizer.encode(label)[0] for label in LABELS]
pd.DataFrame({"label": LABELS, "first_token": [llm.tokenizer.tokenize(label)[0] for label in LABELS],
              "first_token_id": label_ids})

Os três rótulos começam em tokens diferentes, o que permite decidir com uma única posição.

In [ ]:
def classify(text: str) -> str:
    """Escolhe o rótulo de maior logit na primeira posição da resposta."""
    messages = [
        {"role": "system", "content": "Classifique o sentimento da resenha."},
        {"role": "user", "content": text},
    ]
    prompt = llm.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm.tokenizer(prompt, return_tensors="pt").to(llm.weights.device)
    with torch.no_grad():
        logits = llm.weights(**inputs).logits[0, -1]
    return LABELS[int(logits[label_ids].argmax())]

In [ ]:
samples = [
    review,
    "O zíper quebrou em dois dias e o suporte nunca respondeu.",
    "Funciona, nada de especial.",
]
pd.DataFrame({"label": [classify(text) for text in samples], "review": samples})

A saída inválida deixa de existir, porque o conjunto de respostas possíveis é o conjunto de rótulos: a validade passa a ser propriedade do método, e não do comportamento do modelo, e o reprompt deixa de ser necessário. O acerto continua fora de garantia, e a terceira resenha da lista, neutra no texto, é onde conferir isso.

O método decide com o primeiro token. Rótulos que compartilhassem esse token exigiriam comparar a sequência inteira, e um objeto com vários campos exigiria refazer a restrição a cada posição, porque o que é permitido depende do que já foi escrito.

### Outlines

A biblioteca Outlines faz exatamente isso. Ela deriva do esquema um autômato que descreve todas as continuações válidas, e a cada passo zera a probabilidade dos tokens que levariam a uma saída fora do esquema. O modelo continua escolhendo, dentro do que restou.

Outlines entra junto com o `agentkit`, como dependência do pacote. A construção abaixo reaproveita os pesos e o tokenizador já carregados, sem baixar nada de novo.

In [ ]:
from outlines import Generator, from_transformers

guided_model = from_transformers(llm.weights, llm.tokenizer)
generator = Generator(guided_model, Review)

In [ ]:
hard_messages = [{"role": "user", "content": f"Extraia o sentimento, a recomendação e a nota.\n\nResenha: {hard_review}"}]
guided_prompt = llm.tokenizer.apply_chat_template(
    hard_messages, tokenize=False, add_generation_prompt=True
)
print(generator(guided_prompt, max_new_tokens=80, do_sample=False))

A saída é uma string analisável e dentro do esquema, com a mesma instrução escrita à mão que falhava antes. A cerca de bloco de código, o nome de campo escolhido pelo modelo e o sentimento copiado do texto deixam de ser possíveis, porque os tokens que os iniciariam foram zerados.

### generate_structured

A classe `LLM` encapsula essa rota em um método, para que o restante do código receba o objeto validado em vez da string.

In [ ]:
extracted = llm.generate_structured(hard_messages, Review, max_tokens=80)
print(extracted, type(extracted).__name__)

In [ ]:
print(extracted.rating, extracted.sentiment)
print(llm.last_usage)

Por dentro o método faz quatro coisas: aplica o template de conversa quando recebe mensagens, monta o gerador guiado com os pesos já carregados, gera com amostragem desligada e devolve `schema.model_validate_json` sobre o texto produzido. O registro de uso continua sendo preenchido, o que mantém a comparação de custo com as outras rotas.

A garantia é de forma. A nota devolvida está dentro da faixa porque o esquema a impõe, e o valor escolhido continua sendo uma leitura do modelo sobre uma resenha que dizia doze de dez. Validade e verdade são propriedades separadas, e só a segunda exige um conjunto de avaliação com rótulos conferidos por pessoas.

### Exercício 2

Cinco mensagens chegaram na fila de suporte. Escreva o contrato como uma classe `Ticket`, com a categoria, que é `cobrança`, `técnico`, `conta` ou `sugestão`, a urgência, que é um inteiro de 1 a 5 em que 5 significa serviço indisponível para muita gente, e um resumo de uma linha. Escreva `triage_ticket`, que devolve o objeto com `generate_structured`, e junte os cinco em um `DataFrame`.

Toda saída vai passar, porque o esquema não deixa outra coisa acontecer. Aponte então em quais chamados a categoria e a urgência estão erradas, e o que apareceu no campo de resumo que um `str` aceita e a aplicação não deveria.

In [ ]:
TICKETS = [
    "A página de pagamento devolve erro 500 e nenhum cliente consegue finalizar a compra.",
    "Gostaria de saber se o plano anual tem desconto.",
    "Meu cartão foi cobrado três vezes pelo mesmo pedido hoje de manhã.",
    "Seria ótimo se o relatório pudesse ser exportado para o Excel.",
    "Ninguém da minha equipe consegue entrar desde a última atualização.",
]


def triage_ticket(text: str):
    """Classifica o chamado e devolve o objeto validado."""
    return None

## Formas de esquema

O esquema define a tarefa, e trocar a classe troca o problema resolvido sem mudar o laço de chamada. As três formas abaixo cobrem quase tudo que aparece na prática: o campo que pode não existir, o objeto dentro do objeto e a lista de registros resolvida em uma chamada só.

### Campo opcional

A anotação `str | None` autoriza a ausência, e autorizar não é exigir: a máscara aceita tanto o nulo quanto uma string, e sem uma regra de preenchimento o modelo completa o campo faltante com um valor plausível. A regra vai na mensagem `system`, que vale para toda chamada dessa extração.

In [ ]:
class Contact(BaseModel):
    name: str
    email: str | None
    phone: str | None

In [ ]:
EXTRACTOR = (
    "Preencha cada campo apenas com o que está escrito no texto."
    " Use null em todo campo que o texto não informa."
)

contact_text = "Oi, aqui é o Paulo, da equipe de operações; pode falar comigo em paulo@acme.com."
llm.generate_structured(
    [
        {"role": "system", "content": EXTRACTOR},
        {"role": "user", "content": f"Extraia o contato.\n\n{contact_text}"},
    ],
    Contact,
    max_tokens=120,
)

O telefone não está no texto e o campo vem nulo. A instrução não impede o preenchimento quando o dado existe; ela decide o que fazer quando ele falta. As duas camadas têm papéis distintos: o tipo delimita o conjunto de saídas possíveis e a instrução escolhe dentro dele. Quando a saída indesejada precisa ser impossível, e não apenas desincentivada, o caminho é o tipo, com um `Literal` que não inclua texto livre.

### Aninhamento

A forma geral é um esquema dentro do outro. O campo `items` recebe uma lista de `Item`, e cada elemento dessa lista é validado com as regras da classe que o define.

In [ ]:
class Item(BaseModel):
    name: str
    quantity: int = Field(ge=1)
    unit_price: float


class Order(BaseModel):
    customer: str
    items: list[Item]
    urgent: bool

In [ ]:
order_text = (
    "Pedido de Maria Lima: 3 teclados mecânicos a 89,90 cada e 1 monitor a 1299,00. "
    "Ela precisa de tudo antes de sexta."
)
order = llm.generate_structured(
    [{"role": "user", "content": f"Extraia o pedido.\n\n{order_text}"}], Order, max_tokens=200
)
order

In [ ]:
print(order.items[0].name, order.items[0].quantity)
print(sum(item.quantity * item.unit_price for item in order.items))

A lista chegou tipada, e o total sai de uma soma comum sobre objetos. Os preços do texto estão escritos com vírgula e chegaram como `float`, porque o esquema só admite a forma com ponto e a máscara não deixa a vírgula ser escolhida. O prazo mencionado no texto virou `urgent=True`, que é uma leitura do modelo e não um dado copiado.

Cada nível traz as próprias restrições, e a máscara as respeita todas ao mesmo tempo, porque o autômato foi construído a partir do esquema inteiro. O custo aparece no número de tokens, que cresce com a profundidade da estrutura, e no risco de a estrutura ficar grande demais para o orçamento de geração.

### Lote

Um esquema aninhado também é o que permite tratar vários registros em uma chamada, com a lista no topo. A forma aqui não traz nada de novo; o que traz é a decisão de aplicação embutida nela.

In [ ]:
class Triage(BaseModel):
    index: int
    label: Literal["cobrança", "técnico", "conta"]


class Batch(BaseModel):
    results: list[Triage]

In [ ]:
batch_tickets = [
    "Meu cartão foi cobrado duas vezes.",
    "O aplicativo trava ao abrir.",
    "Quero apagar meu perfil.",
]
llm.generate_structured(
    [{"role": "user", "content": "Classifique cada chamado em cobrança, técnico ou conta.\n\n" + "\n".join(batch_tickets)}],
    Batch,
    max_tokens=200,
)

In [ ]:
print(llm.last_usage)

Os três chamados foram resolvidos em uma chamada, com um prompt só e uma passagem de contexto só, o que reduz o custo em relação a três chamadas separadas. O que se paga por isso não aparece no esquema. O alinhamento entre entrada e saída passa a depender do modelo, porque o índice devolvido é um campo como outro qualquer e nada obriga que ele aponte para o chamado certo. Lotes grandes competem com a janela e degradam as últimas linhas.

É o único caso desta seção em que a forma está garantida e a garantia não protege de nada: lotear é uma escolha de aplicação, medida em custo por chamada contra risco de desalinhamento, e o esquema não opina sobre ela.

## Casos comuns

As três formas anteriores descrevem esquemas. Os quatro casos abaixo descrevem usos, e são dos que mais aparecem em aplicação: escolher para onde a mensagem vai, converter texto solto em tipos canônicos, obrigar o modelo a mostrar as contas antes de responder e transformar um parágrafo em uma lista de relações.

### Roteamento

Um sistema com várias rotas precisa decidir qual delas atende cada mensagem. A decisão é um rótulo de um conjunto fechado, e quem consome esse rótulo é o próprio programa.

In [ ]:
class Route(BaseModel):
    destination: Literal["vendas", "suporte", "financeiro"]
    reason: str


DESKS = (
    "vendas cuida de preço, proposta e contrato novo;"
    " suporte cuida de falha técnica e erro no produto;"
    " financeiro cuida de nota fiscal, boleto e pagamento"
)

HANDLERS = {
    "vendas": lambda text: "proposta comercial na fila",
    "suporte": lambda text: "chamado técnico aberto",
    "financeiro": lambda text: "segunda via emitida",
}

In [ ]:
MESSAGES = [
    "Vocês fazem desconto para a compra de 50 licenças?",
    "O sistema não carrega o relatório desde a atualização de ontem.",
    "Preciso da segunda via da nota fiscal de julho.",
]

for message in MESSAGES:
    route = llm.generate_structured(
        [
            {"role": "system", "content": f"Encaminhe a mensagem para o setor responsável: {DESKS}."},
            {"role": "user", "content": message},
        ],
        Route,
        max_tokens=100,
    )
    print(route.destination, "->", HANDLERS[route.destination](message), "|", route.reason[:60])

A linha que importa é `HANDLERS[route.destination]`: um dicionário indexado direto por um valor que veio do modelo. Isso só é seguro porque o `Literal` torna uma chave inexistente irrepresentável; com um campo `str` no lugar, essa linha seria um `KeyError` esperando acontecer, e o código precisaria de um ramo de fallback para toda saída inesperada.

O que o esquema não decide é o acerto. Uma das três mensagens vai para o setor errado, e a garantia continua valendo: a chave existe, o programa roda, o encaminhamento está errado. Repare que a justificativa dessa mensagem fala de desconto, que é assunto de vendas, enquanto o destino escolhido foi outro.

Esse é também o mecanismo de uma chamada de ferramenta, com o esquema no lugar da assinatura da função e o dicionário no lugar do despacho, e é por onde a próxima aula começa.

O campo `reason` merece atenção porque ele não decide nada. Como `destination` vem primeiro no esquema, e o esquema é gerado na ordem em que foi declarado, a justificativa é escrita depois da escolha, e descreve uma decisão já tomada. Inverter os dois campos faz o modelo escrever a justificativa antes de escolher, o que muda o comportamento, e não necessariamente para melhor: vale medir na própria lista acima.

### Normalização

O texto que chega de uma pessoa traz quantidade por extenso, preço com vírgula e data relativa. O que o banco de dados aceita é inteiro, ponto flutuante e data em formato fixo. O esquema é onde essa conversão é declarada, e `Field(pattern=...)` fecha o formato da data.

In [ ]:
class Purchase(BaseModel):
    product: str
    quantity: int
    unit_price: float
    delivery_date: str = Field(pattern=r"\d{4}-\d{2}-\d{2}")


NORMALIZER = (
    "Hoje é segunda-feira, 24 de agosto de 2026."
    " Converta quantidades, valores e datas para os tipos do esquema."
)

ORDERS = [
    "Manda dois teclados a R$ 89,90 cada, entrega na sexta-feira desta semana.",
    "Quero uma dúzia de fones, sai a cento e cinquenta reais a unidade, pode chegar dia 5 do mês que vem.",
    "Três monitores, 1.299,00 cada, precisa estar aqui amanhã.",
]

pd.DataFrame([
    llm.generate_structured(
        [{"role": "system", "content": NORMALIZER}, {"role": "user", "content": text}],
        Purchase,
        max_tokens=120,
    ).model_dump()
    for text in ORDERS
])

As conversões que um punhado de expressões regulares faria estão todas ali: a quantidade por extenso virou inteiro, a vírgula do preço virou ponto e a data relativa virou uma data absoluta no formato do banco. O `pattern` garante o formato da data, então uma resposta como "sexta que vem" não é representável.

E dois valores estão errados. Um preço por extenso perdeu a centena, e a sexta-feira desta semana caiu numa quinta. É a mesma separação de sempre, agora no caso em que ela é mais fácil de esquecer: o formato garantido dá ao dado a aparência de já ter sido conferido.

### Raciocínio em etapas

Um problema com mais de uma conta erra menos quando o modelo escreve as contas antes de responder. O esquema é onde essa exigência fica registrada: uma lista de etapas e, depois delas, a resposta.

In [ ]:
class Step(BaseModel):
    description: str
    result: str


class Reasoning(BaseModel):
    steps: list[Step]
    answer: float

In [ ]:
PROBLEM = (
    "Um pedido tem 12 caixas a R$ 50,00 cada. O frete é R$ 120,00."
    " Há 10% de desconto sobre o valor das caixas, mas não sobre o frete. Qual é o total?"
)
SOLVER = "Resolva em etapas curtas, escritas em português, e só então dê a resposta."

reasoning = llm.generate_structured(
    [{"role": "user", "content": f"{PROBLEM}\n\n{SOLVER}"}],
    Reasoning,
    max_tokens=400,
)

for number, step in enumerate(reasoning.steps, start=1):
    print(number, step.description, "->", step.result)
print("resposta:", reasoning.answer, "| correto:", 12 * 50.00 * 0.90 + 120)

O `answer` é escrito depois de todas as etapas, e não por acaso: os campos são gerados na ordem em que foram declarados, então as contas já estão no contexto quando o número é escolhido. Tirar as etapas do esquema, ou declarar `answer` antes delas, muda a resposta que sai, e é um teste de dois minutos.

O que o esquema obriga é a existência das etapas, não o acerto delas. O que muda é que elas chegam como dado: dá para ler onde a conta saiu do lugar, conferir cada `result` em Python e recusar a resposta quando as etapas não fecham com ela. Uma resposta solta não oferece nada disso.

### Grafo de conhecimento

Extrair um grafo é extrair uma lista de triplas: uma entidade de origem, uma relação e uma entidade de destino. É o mesmo aninhamento do pedido de compra, com a diferença de que aqui a lista é o resultado inteiro.

In [ ]:
class Triple(BaseModel):
    source: str
    relation: str
    target: str


class Graph(BaseModel):
    triples: list[Triple]


PARAGRAPH = (
    "A Vega Logística foi fundada em 2011 por Helena Prado e tem sede em Recife. "
    "Em 2019 a empresa comprou a Rota Norte, que operava no interior do Ceará. "
    "Helena Prado deixou a presidência em 2023 e foi substituída por Caio Bezerra."
)

graph = llm.generate_structured(
    [
        {"role": "system", "content": "Extraia as relações do texto como triplas."},
        {"role": "user", "content": PARAGRAPH},
    ],
    Graph,
    max_tokens=400,
)
pd.DataFrame([triple.model_dump() for triple in graph.triples])

A forma está certa e o vocabulário é inutilizável. Com `relation` livre, o modelo inventa o nome da relação a cada chamada, e o mesmo parágrafo processado amanhã sai com outros nomes. Uma das triplas ainda sai invertida, dizendo que a Vega foi adquirida pela Rota Norte, quando o parágrafo diz o contrário. Um grafo em que a mesma relação aparece escrita de três formas não responde a nenhuma pergunta, porque não dá para juntar duas triplas sem saber que elas falam da mesma coisa. É o mesmo problema da extração livre do começo do notebook, agora nas arestas.

O conserto é o mesmo: fechar o vocabulário no tipo.

In [ ]:
class TypedTriple(BaseModel):
    source: str
    relation: Literal["fundada_em", "fundada_por", "sede_em", "comprou", "presidida_por", "atua_em"]
    target: str


class TypedGraph(BaseModel):
    triples: list[TypedTriple]


knowledge = llm.generate_structured(
    [
        {"role": "system", "content": "Extraia as relações do texto como triplas."},
        {"role": "user", "content": PARAGRAPH},
    ],
    TypedGraph,
    max_tokens=400,
)
pd.DataFrame([triple.model_dump() for triple in knowledge.triples])

Agora as arestas têm nome estável, e duas extrações diferentes podem ser somadas no mesmo grafo. O que continua fora do alcance do esquema é o conteúdo de cada tripla. Um `target` aparece com uma frase no lugar de uma entidade, e as duas presidências entram lado a lado, sem nada que diga que uma substituiu a outra em 2023. Guardar isso exigiria um campo de data em cada tripla, o que é decisão de modelagem, e não de biblioteca.

### Exercício 3

As cinco mensagens abaixo trazem, em texto livre, as notas e a frequência de um aluno, e o regulamento decide o resultado. A tarefa tem resposta certa, e o exercício compara três divisões de trabalho entre o modelo e o código.

Comece pelo gabarito, que é código: `apply_regulation` recebe as três notas e a frequência e devolve o resultado. Todo o resto é medido contra ela.

Depois escreva três contratos para a mesma tarefa. `Direct`, com `average` e `outcome`, em que o modelo faz a conta e aplica a regra. `Reasoned`, com `steps` antes de `average` e `outcome`, usando o `Step` da seção de raciocínio. E `Record`, em que o modelo só extrai o nome, as três notas e a frequência, e quem aplica o regulamento é o `apply_regulation`. Rode os cinco casos nos três e monte um `DataFrame` com o resultado esperado e os três obtidos.

Responda quantos cada contrato acerta, se o erro é de conta ou de regra, e quais casos separam os três. Dois deles foram escritos para isso: um reprova por falta apesar da média suficiente, e outro fica a cinco centésimos da faixa de baixo.

Termine apertando o contrato que ficou em melhor forma. Olhe a única linha em que ele erra, decida qual restrição do esquema teria evitado aquele erro, aplique e meça de novo.

In [ ]:
REGULATION = (
    "A média final é a média ponderada das três avaliações, com pesos 2, 3 e 5."
    " Frequência abaixo de 75% reprova por falta, qualquer que seja a média."
    " Média final de 7 ou mais aprova. Entre 4 e 7, o aluno vai para recuperação."
    " Abaixo de 4, reprova por nota."
)

RECORDS = [
    "Ana Prado tirou 8,0 na primeira avaliação, 7,5 na segunda e 6,4 na terceira, com 92% de frequência.",
    "Bruno Sales foi muito bem no começo, 9,0 e 9,5, mas caiu para 3,0 na terceira; frequência de 88%.",
    "Carla Lima ficou nos 6,0 nas três avaliações e fechou o semestre com 70% de frequência.",
    "Diego Matos tirou 3,0, depois 4,5 e 4,0, sem faltar a nenhuma aula.",
    "Elisa Rocha tirou 10,0 na primeira, 5,0 na segunda e 7,0 na terceira, com 78% de frequência.",
]

OUTCOMES = ["aprovado", "recuperação", "reprovado por nota", "reprovado por falta"]


def apply_regulation(grades: list[float], attendance: float) -> str:
    """Aplica o regulamento e devolve o resultado. É o gabarito do exercício."""
    return ""